# weight-sparsity on vast.ai

Train a TinyStories LM with differentiable weight sparsity
([LTP](https://arxiv.org/abs/2003.00075) / [Continuous Sparsification](https://arxiv.org/abs/1912.04427)).

**Instance setup**

* Template: any recent *PyTorch* image (e.g. `pytorch/pytorch:2.4.0-cuda12.1-cudnn9-runtime`)
  — CUDA and torch are already installed.
* Disk: **≥ 60 GB**. The GPT-Neo-tokenised TinyStories stream is ~1 GB, the HF
  download cache is a few GB, and checkpoints are ~0.6 GB each at the 150M size.
* GPU: a single RTX 4090 / A5000 handles the small config comfortably; use an
  A100 40GB (or reduce `micro_batch_size`) for the ~150M one.

Everything lives under `/workspace`, which is the container volume that
survives instance stop/start (unlike `/tmp` and the image layers). If you
attached a vast.ai **persistent volume** at a different mount point, point
`WORKSPACE` at it in the paths cell.

In [ ]:
!nvidia-smi
!df -h /workspace | tail -1
!free -g | head -2
!nproc

## 1. Paths

In [ ]:
import os

WORKSPACE = '/workspace'                       # persistent volume mount point
REPO_DIR  = f'{WORKSPACE}/weight-sparsity'
DATA_DIR  = f'{WORKSPACE}/data/tinystories'    # tokenised token stream
RUNS_DIR  = f'{WORKSPACE}/runs'                # checkpoints + metrics
CACHE_DIR = f'{WORKSPACE}/hf_cache'            # keep the HF cache off the image layer

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

os.environ['HF_HOME'] = CACHE_DIR
os.environ['HF_DATASETS_CACHE'] = f'{CACHE_DIR}/datasets'
os.environ['PYTHONUNBUFFERED'] = '1'

print('repo :', REPO_DIR)
print('data :', DATA_DIR)
print('runs :', RUNS_DIR)

## 2. Clone the repo and install

In [ ]:
REPO_URL = 'https://github.com/labofdoubt/weight-sparsity.git'

if not os.path.exists(REPO_DIR):
    !git clone -q $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull -q

%cd $REPO_DIR
!git log --oneline -1

In [ ]:
# The PyTorch image already has torch -- install the package without it.
!pip install -q datasets transformers tokenizers pyyaml tqdm
!pip install -q -e . --no-deps

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
print('bf16 supported:', torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

## 3. Prepare the data (once per volume)

Downloads TinyStories, tokenises it and writes flat `uint16` streams to
`$DATA_DIR`. It is skipped automatically if `meta.json` is already there.

`TOKENIZER = 'bpe'` trains an 8k byte-level BPE on TinyStories instead of using
the GPT-Neo vocabulary — the embedding matrix drops from 39M to 6M parameters,
so proportionally more of the model (and more of the maskable weights) sits in
the transformer body.

In [ ]:
TOKENIZER = 'gpt_neo'   # or 'bpe'
NUM_PROC  = 8           # tokenisation workers; keep <= nproc

cmd = (f'python -m wsparse.data --config configs/dense.yaml '
       f'--data.data_dir={DATA_DIR} --data.tokenizer={TOKENIZER} '
       f'--data.tokenizer_path={DATA_DIR}/tokenizer --data.num_proc={NUM_PROC}')
!{cmd}

!ls -lh $DATA_DIR
!cat $DATA_DIR/meta.json

## 4. Choose the run

| config | what it does |
| --- | --- |
| `dense.yaml` | dense baseline |
| `ltp.yaml` | learned per-layer threshold on `w²`, smooth-L0 penalty |
| `cs.yaml` | per-weight gates `s`, smooth-L0 penalty |
| `ltp_target.yaml` / `cs_target.yaml` | same, but driven to a target density |
| `ltp_150m.yaml` / `cs_150m.yaml` | the ~152M-parameter model |

In [ ]:
CONFIG   = 'configs/ltp_150m.yaml'
RUN_NAME = 'ltp_150m_vast'

OVERRIDES = [
    f'--data.data_dir={DATA_DIR}',
    f'--train.out_dir={RUNS_DIR}',
    f'--train.run_name={RUN_NAME}',
    '--train.max_steps=40000',
    '--train.batch_size=128',
    '--train.micro_batch_size=16',      # lower on OOM, raise on a big GPU
    '--train.lr=4e-4',
    '--train.dtype=bfloat16',
    '--train.compile=true',             # worth it for runs this long
    '--train.validate_every_steps=500',
    '--train.checkpoint_every_steps=2000',
    '--train.keep_last_checkpoints=3',
    '--train.sample_every_steps=5000',
    '--train.resume=auto',              # safe to re-run this cell after a restart
    # --- sparsity handles -------------------------------------------------
    '--sparsity.enabled=true',
    '--sparsity.method=ltp',            # ltp | cs
    '--sparsity.targets=["mlp"]',       # ["mlp","attn"] to sparsify attention too
    '--sparsity.beta_schedule=exponential',
    '--sparsity.beta_start=1e4',
    '--sparsity.beta_end=1e6',
    '--sparsity.beta_warmup_steps=2000',
    '--sparsity.mask_lr=1e-6',
    '--sparsity.grad_through_mask=false',
    '--sparsity.l0_coef=0.05',
    # target-density objective instead of / on top of the L0 penalty:
    # '--sparsity.target_density=0.1', '--sparsity.target_density_coef=1.0',
]
OVERRIDE_STR = ' '.join(OVERRIDES)
print(OVERRIDE_STR)

In [ ]:
!python scripts/model_summary.py --config $CONFIG $OVERRIDE_STR

## 5. Train

Two options. **In the foreground** (simple, but the run dies if the notebook
connection drops):

In [ ]:
!python -m wsparse.train --config $CONFIG $OVERRIDE_STR

**Detached** (recommended on vast.ai — survives a browser/kernel disconnect).
Run this cell once, then poll the log cell below as often as you like.

In [ ]:
LOG = f'{RUNS_DIR}/{RUN_NAME}.log'
!mkdir -p $RUNS_DIR
!nohup python -m wsparse.train --config $CONFIG $OVERRIDE_STR > $LOG 2>&1 &
print('started, logging to', LOG)

In [ ]:
!tail -n 25 $LOG

In [ ]:
# stop a detached run
# !pkill -f 'wsparse.train'

## 6. Curves

In [ ]:
import json
import matplotlib.pyplot as plt

path = f'{RUNS_DIR}/{RUN_NAME}/metrics.jsonl'
records = [json.loads(l) for l in open(path)]

def series(key):
    xs = [(r['step'], r[key]) for r in records if key in r]
    return [x for x, _ in xs], [y for _, y in xs]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

s, ce = series('train/ce')
axes[0].plot(s, ce, label='train')
s, v = series('val/ce')
axes[0].plot(s, v, 'o-', label='val (soft)')
s, vh = series('val_hard/ce')
if s:
    axes[0].plot(s, vh, 's--', label='val (hard mask)')
axes[0].set_xlabel('step'); axes[0].set_ylabel('cross-entropy'); axes[0].legend(); axes[0].grid(alpha=.3)

s, d = series('sparsity/density_soft')
if s:
    axes[1].plot(s, d, label='soft (smooth L0)')
    s, dh = series('sparsity/density_hard')
    axes[1].plot(s, dh, label='hard')
    axes[1].set_ylim(0, 1.02)
axes[1].set_xlabel('step'); axes[1].set_ylabel('density'); axes[1].legend(); axes[1].grid(alpha=.3)

s, b = series('sparsity/beta')
if s:
    axes[2].semilogy(s, b, color='tab:red')
    ax2 = axes[2].twinx()
    s2, t = series('sparsity/transition_frac')
    ax2.plot(s2, t, color='tab:gray', alpha=.6)
    ax2.set_ylabel('transition fraction')
axes[2].set_xlabel('step'); axes[2].set_ylabel('beta (log)'); axes[2].grid(alpha=.3)

plt.tight_layout(); plt.show()

## 7. Sample from the checkpoint

In [ ]:
CKPT = f'{RUNS_DIR}/{RUN_NAME}/latest.pt'
!python scripts/generate.py --ckpt "$CKPT" --prompt "Once upon a time" --tokens 200
print()
!python scripts/generate.py --ckpt "$CKPT" --prompt "Once upon a time" --tokens 200 --hard

## 8. Per-layer density

In [ ]:
last = [r for r in records if any(k.startswith('layer_') for k in r)][-1]
for k, v in sorted((k, v) for k, v in last.items() if k.startswith('layer_')):
    print(f'{k[6:]:<28} {v:.4f}')

## 9. Getting the results off the instance

Checkpoints under `/workspace` survive a stop/start of the same instance, but
**not** its destruction. Copy anything you want to keep:

```bash
# from your laptop -- the ssh host/port are on the vast.ai instance card
scp -P <port> -r root@<host>:/workspace/runs/<run_name> ./runs/

# or with the vast CLI
vastai copy <instance_id>:/workspace/runs/<run_name> ./runs/
```

The cell below shrinks a run to just what you usually need (metrics, config and
the final checkpoint) before copying.

In [ ]:
import shutil
src = f'{RUNS_DIR}/{RUN_NAME}'
dst = f'{WORKSPACE}/export/{RUN_NAME}'
os.makedirs(dst, exist_ok=True)
for f in ('metrics.jsonl', 'config.yaml', 'config.json', 'summary.json', 'latest.pt'):
    p = os.path.join(src, f)
    if os.path.exists(p):
        shutil.copy2(p, dst)
!du -sh $dst && ls -lh $dst